<a href="https://colab.research.google.com/github/Anik-Adnan/Python-AI-ML-Masterclass/blob/main/Python/Day08_Mini_Project_Contact_Book.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 08 - Mini Project: Contact Book
## 30-Day  Python Programming Course

| Attribute | Details |
|-----------|---------|
| Day | 8 of 30 |
| Difficulty | Intermediate |
| Topics | Integration project using lists, dicts, and string methods |
| Prerequisites | Day 01-07 |

---

## Learning Objectives
1. Design a small but realistic application using only data structures covered so far (no functions/classes yet - those arrive Day 11+)
2. Apply lists of dictionaries as an in-memory "database"
3. Practice CRUD operations: Create, Read, Update, Delete
4. Implement search, sort, and validation logic using string and list methods
5. Reflect on the limitations of this approach, motivating functions (Day 11) and OOP (Day 13)

---

## Project Specification

We will build a **Contact Book** that stores contacts as dictionaries inside a
list. Each contact has: `name`, `phone`, `email`, `category` (e.g. "family",
"work", "friend").

### Why a List of Dicts?

| Data structure | Why chosen |
|-----------------|------------|
| `list` | Ordered collection of contacts; supports append/remove/sort |
| `dict` (per contact) | Named fields are clearer than tuple positions |
| `set` | Used internally to track unique categories |

This design previews exactly what a relational database row or a JSON API
response looks like - a list of structured records.

---

---
## Part 1 - The Data Store

We use a global list `contacts` as our entire "database" for this lesson.
(On Day 17 we will persist this to a file; on Day 11 we will wrap these
operations in functions; on Day 13+ we will wrap the whole thing in a class.)
---

In [1]:
# ============================================================
# CONTACT BOOK - DATA STORE INITIALISATION
# ============================================================

contacts = [
    {"name": "Alice Johnson", "phone": "555-0101", "email": "alice@example.com",  "category": "work"},
    {"name": "Bob Smith",     "phone": "555-0102", "email": "bob@example.com",    "category": "friend"},
    {"name": "Carol White",   "phone": "555-0103", "email": "carol@example.com",  "category": "family"},
    {"name": "Dave Brown",    "phone": "555-0104", "email": "dave@example.com",   "category": "work"},
]

def print_contacts(contact_list):
    if not contact_list:
        print("  (no contacts)")
        return
    print(f"  {'Name':<18} {'Phone':<12} {'Email':<25} {'Category'}")
    print("  " + "-" * 70)
    for c in contact_list:
        print(f"  {c['name']:<18} {c['phone']:<12} {c['email']:<25} {c['category']}")

print("Initial Contact Book:")
print_contacts(contacts)
print(f"\nTotal contacts: {len(contacts)}")

Initial Contact Book:
  Name               Phone        Email                     Category
  ----------------------------------------------------------------------
  Alice Johnson      555-0101     alice@example.com         work
  Bob Smith          555-0102     bob@example.com           friend
  Carol White        555-0103     carol@example.com         family
  Dave Brown         555-0104     dave@example.com          work

Total contacts: 4


---
## Part 2 - CREATE: Adding New Contacts

We validate input before adding:
- Name must not be empty
- Phone must contain only digits and hyphens
- Email must contain exactly one `@` and at least one `.` after it
---

In [2]:
# ============================================================
# CREATE - Adding New Contacts (with validation)
# ============================================================

def is_valid_phone(phone):
    allowed = set("0123456789-")
    return len(phone) > 0 and all(ch in allowed for ch in phone)

def is_valid_email(email):
    if "@" not in email:
        return False
    local, _, domain = email.partition("@")
    return len(local) > 0 and "." in domain and len(domain.split(".")[-1]) >= 2

def add_contact(name, phone, email, category="general"):
    errors = []
    if not name.strip():
        errors.append("Name cannot be empty")
    if not is_valid_phone(phone):
        errors.append("Phone must contain only digits and hyphens")
    if not is_valid_email(email):
        errors.append("Email format is invalid")
    if any(c["name"].lower() == name.strip().lower() for c in contacts):
        errors.append(f"Contact {name!r} already exists")

    if errors:
        print(f"  FAILED to add {name!r}:")
        for e in errors:
            print(f"    - {e}")
        return False

    contacts.append({
        "name": name.strip(),
        "phone": phone.strip(),
        "email": email.strip().lower(),
        "category": category.strip().lower(),
    })
    print(f"  Added: {name}")
    return True

print("Adding new contacts:")
add_contact("Eve Adams", "555-0105", "eve@example.com", "friend")
add_contact("", "555-0106", "bad@example.com")               # fails: empty name
add_contact("Frank Lee", "555-XYZ", "frank@example.com")      # fails: bad phone
add_contact("Grace Kim", "555-0107", "not-an-email")          # fails: bad email
add_contact("Alice Johnson", "555-9999", "alice2@example.com")  # fails: duplicate

print("\nUpdated Contact Book:")
print_contacts(contacts)

Adding new contacts:
  Added: Eve Adams
  FAILED to add '':
    - Name cannot be empty
  FAILED to add 'Frank Lee':
    - Phone must contain only digits and hyphens
  FAILED to add 'Grace Kim':
    - Email format is invalid
  FAILED to add 'Alice Johnson':
    - Contact 'Alice Johnson' already exists

Updated Contact Book:
  Name               Phone        Email                     Category
  ----------------------------------------------------------------------
  Alice Johnson      555-0101     alice@example.com         work
  Bob Smith          555-0102     bob@example.com           friend
  Carol White        555-0103     carol@example.com         family
  Dave Brown         555-0104     dave@example.com          work
  Eve Adams          555-0105     eve@example.com           friend


---
## Part 3 - READ: Searching and Filtering

We implement three search modes: exact name match, partial (substring) match,
and category filter.
---

In [3]:
# ============================================================
# READ - Searching and Filtering Contacts
# ============================================================

def find_by_name(query):
    query = query.strip().lower()
    return [c for c in contacts if c["name"].lower() == query]

def search_by_substring(query):
    query = query.strip().lower()
    return [c for c in contacts
            if query in c["name"].lower()
            or query in c["email"].lower()]

def filter_by_category(category):
    category = category.strip().lower()
    return [c for c in contacts if c["category"] == category]

print("Exact match for 'Bob Smith':")
print_contacts(find_by_name("Bob Smith"))

print("\nSubstring search for 'al':")
print_contacts(search_by_substring("al"))

print("\nFilter category 'work':")
print_contacts(filter_by_category("work"))

# Build the set of all categories currently in use
categories = {c["category"] for c in contacts}
print(f"\nCategories in use: {sorted(categories)}")
for cat in sorted(categories):
    count = len(filter_by_category(cat))
    print(f"  {cat:<10}: {count} contact(s)")

Exact match for 'Bob Smith':
  Name               Phone        Email                     Category
  ----------------------------------------------------------------------
  Bob Smith          555-0102     bob@example.com           friend

Substring search for 'al':
  Name               Phone        Email                     Category
  ----------------------------------------------------------------------
  Alice Johnson      555-0101     alice@example.com         work

Filter category 'work':
  Name               Phone        Email                     Category
  ----------------------------------------------------------------------
  Alice Johnson      555-0101     alice@example.com         work
  Dave Brown         555-0104     dave@example.com          work

Categories in use: ['family', 'friend', 'work']
  family    : 1 contact(s)
  friend    : 2 contact(s)
  work      : 2 contact(s)


---
## Part 4 - UPDATE: Editing Existing Contacts
---

In [4]:
# ============================================================
# UPDATE - Editing Existing Contacts
# ============================================================

def update_contact(name, **changes):
    matches = find_by_name(name)
    if not matches:
        print(f"  No contact found named {name!r}")
        return False
    contact = matches[0]
    for field, new_value in changes.items():
        if field not in contact:
            print(f"  Unknown field: {field!r}")
            continue
        old_value = contact[field]
        contact[field] = new_value
        print(f"  Updated {contact['name']}.{field}: {old_value!r} -> {new_value!r}")
    return True

print("Updating Bob Smith's phone and category:")
update_contact("Bob Smith", phone="555-9999", category="close friend")

print("\nAttempting to update non-existent contact:")
update_contact("Zoe Nobody", phone="555-0000")

print("\nContact Book after update:")
print_contacts(contacts)

Updating Bob Smith's phone and category:
  Updated Bob Smith.phone: '555-0102' -> '555-9999'
  Updated Bob Smith.category: 'friend' -> 'close friend'

Attempting to update non-existent contact:
  No contact found named 'Zoe Nobody'

Contact Book after update:
  Name               Phone        Email                     Category
  ----------------------------------------------------------------------
  Alice Johnson      555-0101     alice@example.com         work
  Bob Smith          555-9999     bob@example.com           close friend
  Carol White        555-0103     carol@example.com         family
  Dave Brown         555-0104     dave@example.com          work
  Eve Adams          555-0105     eve@example.com           friend


---
## Part 5 - DELETE: Removing Contacts
---

In [5]:
# ============================================================
# DELETE - Removing Contacts
# ============================================================

def delete_contact(name):
    matches = find_by_name(name)
    if not matches:
        print(f"  No contact found named {name!r}")
        return False
    contacts.remove(matches[0])
    print(f"  Deleted: {name}")
    return True

print("Deleting Dave Brown:")
delete_contact("Dave Brown")

print("\nAttempting to delete non-existent contact:")
delete_contact("Nobody Here")

print("\nFinal Contact Book:")
print_contacts(contacts)
print(f"\nTotal contacts remaining: {len(contacts)}")

Deleting Dave Brown:
  Deleted: Dave Brown

Attempting to delete non-existent contact:
  No contact found named 'Nobody Here'

Final Contact Book:
  Name               Phone        Email                     Category
  ----------------------------------------------------------------------
  Alice Johnson      555-0101     alice@example.com         work
  Bob Smith          555-9999     bob@example.com           close friend
  Carol White        555-0103     carol@example.com         family
  Eve Adams          555-0105     eve@example.com           friend

Total contacts remaining: 4


---
## Part 6 - Sorting and Reporting
---

In [6]:
# ============================================================
# SORTING AND REPORTING
# ============================================================

def sorted_contacts(by="name"):
    return sorted(contacts, key=lambda c: c[by].lower())

print("Sorted by name:")
print_contacts(sorted_contacts("name"))

print("\nSorted by category, then name:")
print_contacts(sorted(contacts, key=lambda c: (c["category"], c["name"].lower())))

# Generate a simple text report
print("\n" + "=" * 50)
print("CONTACT BOOK SUMMARY REPORT")
print("=" * 50)
print(f"Total contacts: {len(contacts)}")
categories = {}
for c in contacts:
    categories[c["category"]] = categories.get(c["category"], 0) + 1
print("Breakdown by category:")
for cat, count in sorted(categories.items(), key=lambda x: -x[1]):
    bar = "#" * count
    print(f"  {cat:<12} {count:>2}  {bar}")
print("=" * 50)

Sorted by name:
  Name               Phone        Email                     Category
  ----------------------------------------------------------------------
  Alice Johnson      555-0101     alice@example.com         work
  Bob Smith          555-9999     bob@example.com           close friend
  Carol White        555-0103     carol@example.com         family
  Eve Adams          555-0105     eve@example.com           friend

Sorted by category, then name:
  Name               Phone        Email                     Category
  ----------------------------------------------------------------------
  Bob Smith          555-9999     bob@example.com           close friend
  Carol White        555-0103     carol@example.com         family
  Eve Adams          555-0105     eve@example.com           friend
  Alice Johnson      555-0101     alice@example.com         work

CONTACT BOOK SUMMARY REPORT
Total contacts: 4
Breakdown by category:
  work          1  #
  close friend  1  #
  family    

---
## Exercises

1. Phone Number Formatter
2. Duplicate Detector
3. Bulk Import
4. Contact Merge
5. Export to CSV-like String
---

In [7]:
# EXERCISE 1 - Phone Number Formatter
def format_phone(phone):
    digits = "".join(c for c in phone if c.isdigit())
    if len(digits) == 10:
        return f"({digits[:3]}) {digits[3:6]}-{digits[6:]}"
    elif len(digits) == 7:
        return f"{digits[:3]}-{digits[3:]}"
    return phone  # leave as-is if unrecognised format

test_phones = ["555-0101", "1234567890", "5550101", "555-0102"]
for p in test_phones:
    print(f"  {p:<15} -> {format_phone(p)}")

  555-0101        -> 555-0101
  1234567890      -> (123) 456-7890
  5550101         -> 555-0101
  555-0102        -> 555-0102


In [8]:
# EXERCISE 2 - Duplicate Detector (by email domain analysis)
def domain_report(contact_list):
    domains = {}
    for c in contact_list:
        domain = c["email"].split("@")[-1]
        domains.setdefault(domain, []).append(c["name"])
    return domains

report = domain_report(contacts)
print("Email domains in use:")
for domain, names in report.items():
    print(f"  {domain}: {names}")

Email domains in use:
  example.com: ['Alice Johnson', 'Bob Smith', 'Carol White', 'Eve Adams']


In [9]:
# EXERCISE 3 - Bulk Import
new_contacts_raw = [
    ("Henry Ford",  "555-0201", "henry@example.com",  "work"),
    ("Ivy Chen",    "555-0202", "ivy@example.com",     "friend"),
    ("Jack Ryan",   "BAD-PHONE", "jack@example.com",   "work"),  # invalid
]

print("Bulk importing contacts:")
success_count = 0
for name, phone, email, category in new_contacts_raw:
    if add_contact(name, phone, email, category):
        success_count += 1

print(f"\nImported {success_count}/{len(new_contacts_raw)} contacts successfully")
print_contacts(contacts)

Bulk importing contacts:
  Added: Henry Ford
  Added: Ivy Chen
  FAILED to add 'Jack Ryan':
    - Phone must contain only digits and hyphens

Imported 2/3 contacts successfully
  Name               Phone        Email                     Category
  ----------------------------------------------------------------------
  Alice Johnson      555-0101     alice@example.com         work
  Bob Smith          555-9999     bob@example.com           close friend
  Carol White        555-0103     carol@example.com         family
  Eve Adams          555-0105     eve@example.com           friend
  Henry Ford         555-0201     henry@example.com         work
  Ivy Chen           555-0202     ivy@example.com           friend


In [10]:
# EXERCISE 4 - Contact Merge (combine two contact lists, skip exact dupes)
team_a = [
    {"name": "Alice Johnson", "phone": "555-0101", "email": "alice@example.com", "category": "work"},
    {"name": "Karen Hill",    "phone": "555-0301", "email": "karen@example.com", "category": "work"},
]
team_b = [
    {"name": "Karen Hill",    "phone": "555-0301", "email": "karen@example.com", "category": "work"},
    {"name": "Leo Park",      "phone": "555-0302", "email": "leo@example.com",   "category": "work"},
]

def merge_contact_lists(list1, list2):
    merged = list1.copy()
    existing_names = {c["name"].lower() for c in merged}
    added = 0
    for c in list2:
        if c["name"].lower() not in existing_names:
            merged.append(c)
            existing_names.add(c["name"].lower())
            added += 1
    return merged, added

merged, added = merge_contact_lists(team_a, team_b)
print(f"Merged list ({added} new contacts added):")
print_contacts(merged)

Merged list (1 new contacts added):
  Name               Phone        Email                     Category
  ----------------------------------------------------------------------
  Alice Johnson      555-0101     alice@example.com         work
  Karen Hill         555-0301     karen@example.com         work
  Leo Park           555-0302     leo@example.com           work


In [11]:
# EXERCISE 5 - Export to CSV-like String
def export_to_csv_string(contact_list):
    lines = ["name,phone,email,category"]
    for c in contact_list:
        lines.append(f"{c['name']},{c['phone']},{c['email']},{c['category']}")
    return "\n".join(lines)

csv_output = export_to_csv_string(contacts)
print(csv_output)

def import_from_csv_string(csv_text):
    lines = csv_text.strip().split("\n")
    header = lines[0].split(",")
    records = []
    for line in lines[1:]:
        values = line.split(",")
        records.append(dict(zip(header, values)))
    return records

reimported = import_from_csv_string(csv_output)
print(f"\nRe-imported {len(reimported)} records, matches original: "
      f"{reimported == contacts}")

name,phone,email,category
Alice Johnson,555-0101,alice@example.com,work
Bob Smith,555-9999,bob@example.com,close friend
Carol White,555-0103,carol@example.com,family
Eve Adams,555-0105,eve@example.com,friend
Henry Ford,555-0201,henry@example.com,work
Ivy Chen,555-0202,ivy@example.com,friend

Re-imported 6 records, matches original: True


---
## Challenge Problems

1. Fuzzy Name Search (Levenshtein-lite)
2. Contact Book Statistics Dashboard
---

In [12]:
# CHALLENGE 1 - Simple Fuzzy Name Search
def simple_edit_distance(a, b):
    a, b = a.lower(), b.lower()
    if len(a) < len(b):
        return simple_edit_distance(b, a)
    if len(b) == 0:
        return len(a)
    previous_row = list(range(len(b) + 1))
    for i, ca in enumerate(a):
        current_row = [i + 1]
        for j, cb in enumerate(b):
            insertions = previous_row[j + 1] + 1
            deletions  = current_row[j] + 1
            substitutions = previous_row[j] + (ca != cb)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    return previous_row[-1]

def fuzzy_search(query, contact_list, max_distance=3):
    results = []
    for c in contact_list:
        dist = simple_edit_distance(query, c["name"])
        if dist <= max_distance:
            results.append((c, dist))
    results.sort(key=lambda x: x[1])
    return results

queries = ["Alise Jonson", "Bobby Smyth", "Carol Wite"]
for q in queries:
    matches = fuzzy_search(q, contacts, max_distance=4)
    print(f"\nSearching for {q!r}:")
    for c, dist in matches[:3]:
        print(f"  {c['name']:<18} (edit distance={dist})")


Searching for 'Alise Jonson':
  Alice Johnson      (edit distance=2)

Searching for 'Bobby Smyth':
  Bob Smith          (edit distance=3)

Searching for 'Carol Wite':
  Carol White        (edit distance=1)


In [13]:
# CHALLENGE 2 - Contact Book Statistics Dashboard
def dashboard(contact_list):
    print("=" * 55)
    print("CONTACT BOOK DASHBOARD".center(55))
    print("=" * 55)
    print(f"Total contacts        : {len(contact_list)}")

    categories = {}
    domains = {}
    name_lengths = []
    for c in contact_list:
        categories[c["category"]] = categories.get(c["category"], 0) + 1
        domain = c["email"].split("@")[-1]
        domains[domain] = domains.get(domain, 0) + 1
        name_lengths.append(len(c["name"]))

    print(f"Unique categories     : {len(categories)}")
    print(f"Unique email domains  : {len(domains)}")
    if name_lengths:
        print(f"Avg name length       : {sum(name_lengths)/len(name_lengths):.1f} chars")
        longest = max(contact_list, key=lambda c: len(c["name"]))
        print(f"Longest name          : {longest['name']!r}")

    print("\nCategory breakdown:")
    for cat, count in sorted(categories.items(), key=lambda x: -x[1]):
        pct = count / len(contact_list) * 100
        bar = "#" * int(pct / 4)
        print(f"  {cat:<12} {count:>2} ({pct:>5.1f}%) {bar}")
    print("=" * 55)

dashboard(contacts)

                 CONTACT BOOK DASHBOARD                
Total contacts        : 6
Unique categories     : 4
Unique email domains  : 1
Avg name length       : 10.0 chars
Longest name          : 'Alice Johnson'

Category breakdown:
  work          2 ( 33.3%) ########
  friend        2 ( 33.3%) ########
  close friend  1 ( 16.7%) ####
  family        1 ( 16.7%) ####


---
## Summary - Day 08

| Concept Reinforced | How It Was Used |
|---------------------|------------------|
| List of dicts | Modeled the entire contact database |
| String validation | Phone and email format checking |
| List comprehensions | Search and filter operations |
| Set | Tracking unique categories |
| sorted() with key | Multiple sort criteria (tuple keys) |
| Dictionary `.get()` and `setdefault()` | Category and domain counting |

### Reflection: What's Missing?

This project works, but notice the limitations:
- All logic is written as loose functions operating on a global list - fragile
- No persistence: data vanishes when the notebook restarts (Day 17 fixes this)
- No encapsulation: anyone can directly mutate `contacts` and break invariants
  (Day 13-16 OOP will wrap this safely inside a `ContactBook` class)
- No error handling beyond manual validation (Day 19 covers exceptions properly)

These limitations are exactly the motivation for what comes next.

**Next: Day 09 - Conditional Statements (if-elif-else)**

---